# OpenPronounce

Open-source, phoneme-level English pronunciation assessment.

Give it a recording and the sentence the speaker meant to say; it returns a 0-100 score, the list of mispronounced words (expected vs heard phonemes, in IPA), the transcription and the prosody (pitch and energy) curves.

Under the hood: a Wav2Vec2 model fine-tuned to output espeak phones recognizes what was actually said, which is aligned with the phones expected for the sentence; a second Wav2Vec2 model gives the transcription, and DTW on embeddings against a synthetic reference gives an acoustic distance. See the [blog post](https://blog.lepine.pro/en/ai-wav2vec-pronunciation-vectorization/) for the approach.

> On Colab: `Runtime > Run all`. The first cell takes about two minutes (installs torch and downloads two 1.2 GB models).

In [ ]:
# Install espeak-ng (phonemization) and OpenPronounce. Takes ~2 min on Colab.
!apt-get -qq install -y espeak-ng ffmpeg > /dev/null
!pip install -q openpronounce

In [ ]:
# Load an audio file as a 16 kHz mono waveform (mp3, wav, flac, ogg... all work)
import os
import urllib.request

from openpronounce import audio, speech

audio_path = "assets/example.mp3"
if not os.path.exists(audio_path):
    os.makedirs("assets", exist_ok=True)
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Halleck45/OpenPronounce/main/assets/example.mp3", audio_path
    )

sound = audio.load(audio_path)

from IPython.display import Audio
Audio(sound, rate=16000)

## Pronunciation

The recording above says *"Hello, how are you?"* with a strong accent. Let's score it.

The model we use is Wav2Vec2 (`facebook/wav2vec2-large-960h`), trained on 960 h of read English. It works well on adult speech; children's voices and heavy background noise degrade the results.

In [ ]:
prediction = speech.compare_audio_with_text(sound, "Hello, how are you?")

print("Score:", prediction["score"], "/ 100")
print("Heard:", prediction["transcribe"])

In [ ]:
# What the phone recognizer heard, then the word-level feedback (expected vs heard phones, IPA)
print("Heard phones:", " ".join(prediction["differences"]["heard_phones"]))
print()
for error in prediction["differences"]["errors"]:
    heard = f"/{error['actual']}/" if error["actual"] else "(missing)"
    print(f"{error['word']:12s} expected /{error['expected']}/  heard {heard}")

## Try your own sentence

Record yourself (any phone voice memo will do), upload the file to Colab, then change the path and the sentence below.

In [ ]:
# from google.colab import files; uploaded = files.upload()   # uncomment on Colab
# my_audio = audio.load(next(iter(uploaded)))
# my_prediction = speech.compare_audio_with_text(my_audio, "The sentence I just read")
# print(my_prediction["score"], my_prediction["differences"]["words_with_errors"])

## Prosody

Prosody is the rhythm and intonation of speech. OpenPronounce returns two contours: the energy (loudness) and the fundamental frequency F0 (pitch).

In [ ]:
import matplotlib.pyplot as plt

prosody = prediction["prosody"]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 5), sharex=False)
ax1.plot(prosody["energy"]); ax1.set_title("Energy (loudness)")
ax2.plot(prosody["f0"]); ax2.set_title("F0 (pitch, Hz)")
plt.tight_layout(); plt.show()

## Other languages

The phone recognizer is multilingual, so French, Spanish, German, Italian, Portuguese and Dutch work with `lang=`. Below we synthesize a French reference (this needs the network for gTTS) and score it against itself, then against the wrong sentence. The first call downloads the French transcription model (~1.2 GB).

In [ ]:
from openpronounce import LANGUAGES
print("Supported:", ", ".join(LANGUAGES))

sentence = "Bonjour, comment allez-vous ?"
french_audio = audio.load(audio.text2speech(sentence, lang="fr"))

good = speech.compare_audio_with_text(french_audio, sentence, lang="fr")
wrong = speech.compare_audio_with_text(french_audio, "Bonsoir, tu es médecin", lang="fr")
print("Same sentence :", good["score"], good["differences"]["words_with_errors"])
print("Wrong sentence:", wrong["score"], wrong["differences"]["words_with_errors"])

## Under the hood: phonemes and alignment

You can call the lower-level building blocks directly.

In [ ]:
print(speech.get_phonemes("Hello, how are you?"))
print(speech.compare_transcriptions("hello who are you", "hello how are you")["errors"])